In [ ]:
%pip install pymc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import matplotlib.dates as mdates
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
import pymc as pm
import arviz as az

# Set plotting style
plt.style.use('ggplot')
sns.set_palette("deep")

# Set random seeds for reproducibility
np.random.seed(42)

# Define a function to load data
def load_and_prepare_data(file_path):
    """
    Load and prepare taxi data from a parquet file
    
    Parameters:
    file_path: Path to the parquet file
    
    Returns:
    df: Prepared DataFrame
    """
    df = pd.read_parquet(file_path)
    
    # Basic data cleaning and preparation
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
    df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
    
    # Extract datetime components
    df['pickup_date'] = df['tpep_pickup_datetime'].dt.date
    df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
    df['pickup_day'] = df['tpep_pickup_datetime'].dt.day
    df['pickup_month'] = df['tpep_pickup_datetime'].dt.month
    df['pickup_year'] = df['tpep_pickup_datetime'].dt.year
    df['pickup_weekday'] = df['tpep_pickup_datetime'].dt.weekday
    
    # Remove outliers - adjust these filters based on your data
    df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]  # Remove unrealistic distances
    df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]      # Remove unrealistic fares
    
    return df

# Load and prepare data
df = load_and_prepare_data('taxi-dataset.parquet')
print(df.columns)
print(df.head())

print(f"Data range: {df['tpep_pickup_datetime'].min()} to {df['tpep_pickup_datetime'].max()}")

# Get 2022 data
df_2022 = df[df['tpep_pickup_datetime'].dt.year == 2022]
print("Sample of 2022 data:")
print(df_2022.head())

# Calculate fare rate
df['fare_rate'] = df['fare_amount'] / df['trip_distance']
df['fare_rate'] = df['fare_rate'].replace([np.inf, -np.inf], np.nan).fillna(df['fare_rate'].median())

# Ensure data is sorted by datetime
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df = df.sort_values('tpep_pickup_datetime')

# Set pickup_datetime as index for time series analysis
df_time = df.set_index('tpep_pickup_datetime').copy()

# Create different aggregations for time series analysis
# Daily average fare rate
daily_fare_rate = df_time.resample('D')['fare_rate'].mean()

# Hourly average fare rate
hourly_fare_rate = df_time.resample('H')['fare_rate'].mean()

# Weekly average fare rate
weekly_fare_rate = df_time.resample('W')['fare_rate'].mean()

# Monthly average fare rate
monthly_fare_rate = df_time.resample('M')['fare_rate'].mean()

# Fill NaN values with interpolation for analysis
daily_fare_rate = daily_fare_rate.interpolate()
hourly_fare_rate = hourly_fare_rate.interpolate()
weekly_fare_rate = weekly_fare_rate.interpolate()
monthly_fare_rate = monthly_fare_rate.interpolate()

# Function to perform stationarity test
def stationarity_test(time_series, title):
    """
    Perform Augmented Dickey-Fuller test to check for stationarity
    
    Parameters:
    time_series: The time series to test
    title: Title for the test output
    """
    print(f"\nStationarity Test for {title}")
    result = adfuller(time_series.dropna())
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'\t{key}: {value:.4f}')
    
    if result[1] < 0.05:
        print(f"Result: {title} is stationary (p-value < 0.05)")
    else:
        print(f"Result: {title} is not stationary (p-value >= 0.05)")
    print("-" * 50)

# Perform stationarity tests
for ts, name in [
    (daily_fare_rate, "Daily Average Fare Rate"),
    (hourly_fare_rate, "Hourly Average Fare Rate"),
    (weekly_fare_rate, "Weekly Average Fare Rate"),
    (monthly_fare_rate, "Monthly Average Fare Rate")
]:
    stationarity_test(ts, name)

# Function to perform Bayesian structural time series decomposition
def bsts_decomposition(time_series, title, seasonal_periods=None, n_samples=1000, tune=1000):
    """
    Decompose time series using Bayesian structural time series model with PyMC
    
    Parameters:
    time_series: The time series to decompose
    title: Title for plots
    seasonal_periods: List of seasonal periods to include
    n_samples: Number of posterior samples
    tune: Number of tuning samples
    
    Returns:
    idata: InferenceData object with posterior samples
    components: Dict of component posterior means
    """
    print(f"\nPerforming Bayesian Structural Time Series Decomposition for {title}")
    
    # Convert to numpy array
    y = time_series.values
    n = len(y)
    
    # Standardize data for better sampling
    y_mean = np.mean(y)
    y_std = np.std(y)
    y_standardized = (y - y_mean) / y_std
    
    # Prepare time indices
    t = np.arange(n)
    
    # Set up PyMC model
    with pm.Model() as model:
        # Level (local level) component priors
        sigma_level = pm.HalfNormal("sigma_level", sigma=0.1)
        level_init = pm.Normal("level_init", mu=0, sigma=1)
        
        # Random walk for the level component
        level_innovations = pm.Normal("level_innovations", mu=0, sigma=sigma_level, shape=n-1)
        level = pm.Deterministic("level", pm.math.concatenate([[level_init], level_init + pm.math.cumsum(level_innovations)]))
        
        # Trend (slope) component priors if needed
        if True:  # Set to False to disable trend
            sigma_trend = pm.HalfNormal("sigma_trend", sigma=0.01)
            trend_init = pm.Normal("trend_init", mu=0, sigma=0.1)
            
            # Random walk for trend innovations
            trend_innovations = pm.Normal("trend_innovations", mu=0, sigma=sigma_trend, shape=n-1)
            trend = pm.Deterministic("trend", pm.math.concatenate([[trend_init], trend_init + pm.math.cumsum(trend_innovations)]))
        else:
            trend = pm.Deterministic("trend", pm.math.zeros(n))
        
        # Define seasonal components if needed
        seasonal_components = []
        if seasonal_periods:
            for i, period in enumerate(seasonal_periods):
                # Fourier terms for seasonality (more flexible than dummy variables)
                n_fourier_terms = min(period // 2, 10)  # Cap at 10 to avoid overfitting
                
                # Fourier series for seasonal components
                fourier_terms = np.zeros((n, 2 * n_fourier_terms))
                for j in range(n_fourier_terms):
                    p = period / (j + 1)
                    fourier_terms[:, 2*j] = np.sin(2 * np.pi * t / p)
                    fourier_terms[:, 2*j+1] = np.cos(2 * np.pi * t / p)
                
                # Prior for seasonal coefficients
                seasonal_coef_name = f"seasonal_coef_{period}"
                seasonal_coef = pm.Normal(seasonal_coef_name, mu=0, sigma=0.1, shape=2*n_fourier_terms)
                
                # Seasonal component
                seasonal_name = f"seasonal_{period}"
                seasonal = pm.Deterministic(seasonal_name, pm.math.dot(fourier_terms, seasonal_coef))
                seasonal_components.append(seasonal)
        
        # Sum up all seasonal components
        if seasonal_components:
            seasonality = pm.Deterministic("seasonality", sum(seasonal_components))
        else:
            seasonality = pm.Deterministic("seasonality", pm.math.zeros(n))
        
        # Observation noise
        sigma_obs = pm.HalfNormal("sigma_obs", sigma=0.1)
        
        # Expected value: trend + level + seasonality
        mu = level + trend + seasonality
        
        # Likelihood
        likelihood = pm.Normal("y", mu=mu, sigma=sigma_obs, observed=y_standardized)
        
        # Sample from the posterior
        idata = pm.sample(n_samples, tune=tune, return_inferencedata=True)
    
    # Calculate posterior means for components
    posterior_means = {
        "level": idata.posterior["level"].mean(dim=["chain", "draw"]).values * y_std + y_mean,
        "trend": idata.posterior["trend"].mean(dim=["chain", "draw"]).values * y_std,
        "seasonality": idata.posterior["seasonality"].mean(dim=["chain", "draw"]).values * y_std,
    }
    
    # Calculate fitted values and residuals
    posterior_means["fitted"] = (posterior_means["level"] + posterior_means["trend"] + 
                                posterior_means["seasonality"])
    posterior_means["residuals"] = y - posterior_means["fitted"]
    
    # Create plots of components with credible intervals
    _, axs = plt.subplots(4, 1, figsize=(14, 16))
    
    # Original data with fitted values
    axs[0].plot(time_series.index, y, label="Observed", alpha=0.7)
    axs[0].plot(time_series.index, posterior_means["fitted"], label="Fitted", color="red")
    
    # Add 95% credible intervals for the fitted values - THE FIX IS HERE
    # Calculate the CI for each time point separately to ensure we have 1D arrays
    fitted_samples = idata.posterior["level"] + idata.posterior["trend"] + idata.posterior["seasonality"]
    fitted_samples = fitted_samples.values * y_std + y_mean
    
    # Extract lower and upper CI bounds for each time point
    fitted_lower = np.percentile(fitted_samples, 2.5, axis=(0, 1))
    fitted_upper = np.percentile(fitted_samples, 97.5, axis=(0, 1))
    
    # Make sure these are 1D arrays matching the length of time_series.index
    if fitted_lower.ndim > 1 or fitted_upper.ndim > 1:
        # If they're still multi-dimensional, flatten them
        fitted_lower = fitted_lower.flatten()[:len(time_series.index)]
        fitted_upper = fitted_upper.flatten()[:len(time_series.index)]
    
    # Now fill_between should work with 1D arrays
    axs[0].fill_between(time_series.index, fitted_lower, fitted_upper, color="red", alpha=0.2)
    
    axs[0].set_title(f"{title} - Observed vs Fitted")
    axs[0].legend()
    axs[0].grid(True)
    
    # Level component - apply the same fix for CI plotting
    level_samples = idata.posterior["level"].values * y_std + y_mean
    level_lower = np.percentile(level_samples, 2.5, axis=(0, 1))
    level_upper = np.percentile(level_samples, 97.5, axis=(0, 1))
    
    # Ensure 1D arrays
    if level_lower.ndim > 1 or level_upper.ndim > 1:
        level_lower = level_lower.flatten()[:len(time_series.index)]
        level_upper = level_upper.flatten()[:len(time_series.index)]
        
    axs[1].plot(time_series.index, posterior_means["level"], label="Level", color="green")
    axs[1].fill_between(time_series.index, level_lower, level_upper, color="green", alpha=0.2)
    axs[1].set_title("Level Component")
    axs[1].grid(True)
    
    # Trend component - apply the same fix for CI plotting
    trend_samples = idata.posterior["trend"].values * y_std
    trend_lower = np.percentile(trend_samples, 2.5, axis=(0, 1))
    trend_upper = np.percentile(trend_samples, 97.5, axis=(0, 1))
    
    # Ensure 1D arrays
    if trend_lower.ndim > 1 or trend_upper.ndim > 1:
        trend_lower = trend_lower.flatten()[:len(time_series.index)]
        trend_upper = trend_upper.flatten()[:len(time_series.index)]
        
    axs[2].plot(time_series.index, posterior_means["trend"], label="Trend", color="blue")
    axs[2].fill_between(time_series.index, trend_lower, trend_upper, color="blue", alpha=0.2)
    axs[2].set_title("Trend Component")
    axs[2].grid(True)
    
    # Seasonal component - apply the same fix for CI plotting
    seasonal_samples = idata.posterior["seasonality"].values * y_std
    seasonal_lower = np.percentile(seasonal_samples, 2.5, axis=(0, 1))
    seasonal_upper = np.percentile(seasonal_samples, 97.5, axis=(0, 1))
    
    # Ensure 1D arrays
    if seasonal_lower.ndim > 1 or seasonal_upper.ndim > 1:
        seasonal_lower = seasonal_lower.flatten()[:len(time_series.index)]
        seasonal_upper = seasonal_upper.flatten()[:len(time_series.index)]
        
    axs[3].plot(time_series.index, posterior_means["seasonality"], label="Seasonality", color="purple")
    axs[3].fill_between(time_series.index, seasonal_lower, seasonal_upper, color="purple", alpha=0.2)
    axs[3].set_title("Seasonal Component")
    axs[3].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Plot residuals
    plt.figure(figsize=(14, 6))
    plt.plot(time_series.index, posterior_means["residuals"], label="Residuals", color="gray")
    plt.title(f"{title} - Residuals")
    plt.grid(True)
    plt.show()
    
    # Calculate and print component statistics
    component_variances = {
        "level": np.var(posterior_means["level"]),
        "trend": np.var(posterior_means["trend"]),
        "seasonality": np.var(posterior_means["seasonality"]),
        "residuals": np.var(posterior_means["residuals"]),
        "total": np.var(y)
    }
    
    # Calculate percentage of variance explained by each component
    variance_explained = {
        k: v / component_variances["total"] * 100 
        for k, v in component_variances.items() if k != "total"
    }
    
    print(f"\nComponent Analysis for {title}:")
    print(f"Total Variance: {component_variances['total']:.4f}")
    for component, explained in variance_explained.items():
        print(f"{component.capitalize()}: {explained:.2f}% of total variance")
    
    # Calculate model fit statistics
    mse = np.mean(posterior_means["residuals"] ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(posterior_means["residuals"]))
    
    print(f"\nModel Fit Statistics:")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print("-" * 50)
    
    # Check for autocorrelation in residuals
    plt.figure(figsize=(14, 6))
    
    # Calculate autocorrelation for multiple lags
    residual_series = pd.Series(posterior_means["residuals"])
    lags = range(1, min(51, len(residual_series) // 5))  # Use up to 50 lags or 1/5 of series length
    
    # Calculate autocorrelation for each lag
    autocorr_values = [residual_series.autocorr(lag=lag) for lag in lags]
    
    # Plot autocorrelations
    plt.bar(lags, autocorr_values)
    plt.title(f"{title} - Residual Autocorrelation")
    plt.xlabel("Lag")
    plt.ylabel("Autocorrelation")
    plt.axhline(y=0, color='gray', linestyle='-')
    
    # Add confidence bands at 95% (approximately ±2/√n)
    conf_level = 1.96 / np.sqrt(len(residual_series))
    plt.axhline(y=conf_level, color='r', linestyle='--', alpha=0.7)
    plt.axhline(y=-conf_level, color='r', linestyle='--', alpha=0.7)
    
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    return idata, posterior_means

# Apply Bayesian decomposition to different time scales
print("\nApplying Bayesian structural time series decomposition...")

# For computational efficiency, use a subset of data
# Adjust these samples based on your computational resources
hourly_sample = hourly_fare_rate[:24*14]  # 14 days of hourly data
daily_sample = daily_fare_rate[:90]  # 90 days of daily data
weekly_sample = weekly_fare_rate  # All weekly data
monthly_sample = monthly_fare_rate  # All monthly data

# Daily decomposition with weekly seasonality
daily_idata, daily_components = bsts_decomposition(
    daily_sample,
    "Daily Average Fare Rate",
    seasonal_periods=[7],  # Weekly seasonality
    n_samples=500,  # Reduced for speed
    tune=500
)

# Hourly decomposition with daily seasonality
hourly_idata, hourly_components = bsts_decomposition(
    hourly_sample,
    "Hourly Average Fare Rate",
    seasonal_periods=[24],  # Daily seasonality
    n_samples=500,
    tune=500
)

# Weekly decomposition with yearly seasonality (if enough data)
if len(weekly_sample) >= 52:
    weekly_idata, weekly_components = bsts_decomposition(
        weekly_sample,
        "Weekly Average Fare Rate",
        seasonal_periods=[52],  # Yearly seasonality
        n_samples=500,
        tune=500
    )
else:
    print("Not enough weekly data for yearly seasonality analysis")

# Monthly decomposition with yearly seasonality (if enough data)
if len(monthly_sample) >= 12:
    monthly_idata, monthly_components = bsts_decomposition(
        monthly_sample,
        "Monthly Average Fare Rate",
        seasonal_periods=[12],  # Yearly seasonality
        n_samples=500,
        tune=500
    )
else:
    print("Not enough monthly data for yearly seasonality analysis")

# Function to analyze seasonal patterns more deeply
def analyze_seasonal_patterns(time_series, title, period):
    """
    Analyze seasonal patterns by grouping and averaging data
    
    Parameters:
    time_series: Time series with DatetimeIndex
    title: Title for plots
    period: 'day', 'week', 'month', or 'year'
    """
    print(f"\nAnalyzing {period}ly patterns for {title}")
    
    if period == 'day':
        # Group by hour of day
        grouped = time_series.groupby(time_series.index.hour).mean()
        x_label = 'Hour of Day'
        xticks = np.arange(24)
    elif period == 'week':
        # Group by day of week
        grouped = time_series.groupby(time_series.index.dayofweek).mean()
        x_label = 'Day of Week'
        xticks = np.arange(7)
        plt.xticks(xticks, ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
    elif period == 'month':
        # Group by day of month
        grouped = time_series.groupby(time_series.index.day).mean()
        x_label = 'Day of Month'
        xticks = np.arange(1, 32)
    elif period == 'year':
        # Group by month of year
        grouped = time_series.groupby(time_series.index.month).mean()
        x_label = 'Month of Year'
        xticks = np.arange(1, 13)
        plt.xticks(xticks, ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
    else:
        raise ValueError("Period must be 'day', 'week', 'month', or 'year'")
    
    # Plot the pattern
    plt.figure(figsize=(14, 6))
    plt.plot(grouped.index, grouped.values, 'o-', linewidth=2, markersize=8)
    plt.title(f"{title} - Average by {x_label}")
    plt.xlabel(x_label)
    plt.ylabel('Average Value')
    plt.grid(True)
    plt.xticks(xticks)
    plt.tight_layout()
    plt.show()
    
    return grouped

# Analyze seasonal patterns for different time scales
print("\nAnalyzing seasonal patterns...")

# Daily patterns for hourly data
daily_pattern = analyze_seasonal_patterns(hourly_fare_rate, "Hourly Fare Rate", "day")

# Weekly patterns for daily data
weekly_pattern = analyze_seasonal_patterns(daily_fare_rate, "Daily Fare Rate", "week")

# Monthly patterns for daily data (if enough data spans multiple months)
if daily_fare_rate.index.max() - daily_fare_rate.index.min() >= pd.Timedelta(days=60):
    monthly_pattern = analyze_seasonal_patterns(daily_fare_rate, "Daily Fare Rate", "month")

# Yearly patterns for monthly data (if enough data spans multiple years)
if monthly_fare_rate.index.max().year - monthly_fare_rate.index.min().year >= 1:
    yearly_pattern = analyze_seasonal_patterns(monthly_fare_rate, "Monthly Fare Rate", "year")

# Additional analysis: Fare rate by hour of day
fare_by_hour = df.groupby('pickup_hour')['fare_rate'].mean()

plt.figure(figsize=(12, 6))
sns.barplot(x=fare_by_hour.index, y=fare_by_hour.values)
plt.title('Average Fare Rate by Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Average Fare Rate ($/mile)')
plt.xticks(range(0, 24))
plt.grid(True, axis='y')
plt.show()

# Fare rate vs. trip distance
plt.figure(figsize=(12, 6))
plt.scatter(df['trip_distance'], df['fare_rate'], alpha=0.3)
plt.title('Fare Rate vs. Trip Distance')
plt.xlabel('Trip Distance (miles)')
plt.ylabel('Fare Rate ($/mile)')
plt.grid(True)
plt.show()

print("Bayesian structural time series analysis complete!")

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 3.3 MB/s eta 0:00:00ta 0:00:01
  Created wheel for cons: filename=cons-0.4.6-py3-none-any.whl size=9100 sha256=b29e51636cb67cdcd2878d731267e2e71574a11a483981126ec266c13e298d3d
  Stored in directory: /Users/tanmay/Library/Caches/pip/wheels/95/8f/45/fe0a5b5e232401da571d514eb545833fbe220993ac8336c94e
  Created wheel for logical-unification: filename=logical_unification-0.4.6-py3-none-any.whl size=13911 sha256=0b369bed0326afbe486e268ad1583b7c07641c6f4b5eafd0224a965223bd497f
  Stored in directory: /Users/tanmay/Library/Caches/pip/wheels/b8/34/a9/c11a21ef1f1b6d2e5ae518dd5d28c0bd2b131c5d6e5d4417c3
  Created wheel for etuples: filename=etuples-0.3.9-py3-none-any.whl size=12618 sha256=9a7aeb3139fd06dfabf1dce5f7bdf6576ba8cda9998978f2dec3246fe796ea94
  Stored in directory: /Users/t

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma_level, level_init, level_innovations, sigma_trend, trend_init, trend_innovations, seasonal_coef_7, sigma_obs]


Output()

Sampling 4 chains for 500 tune and 500 draw iterations (2_000 + 2_000 draws total) took 29 seconds.
There were 897 divergences after tuning. Increase `target_accept` or reparameterize.
Chain 0 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 3 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
